# 02 — Brain MRI Classification : carnet d'experimentation pre-production

**Auteur :** smatt  
**Date de travail :** Mars 2026  
**Dataset cible :** Brain Tumor MRI Dataset  
**Objectif :** etablir une baseline multiclasse reproductible avant comparaison architecturale et exposition applicative

## Question de travail

Ce notebook sert a verifier si une classification `glioma` / `meningioma` / `pituitary` / `no_tumor` produit un signal utile et stable, ou si la difficulte principale vient d'un chevauchement entre classes, d'un desequilibre des donnees ou d'une preparation insuffisante des images.

## Hypotheses a verifier

- les quatre classes sont suffisamment separees pour justifier une tete softmax simple en premiere passe ;
- les erreurs importantes se concentrent sur quelques couples de classes et non sur un effondrement global du modele ;
- le desequilibre de classes peut etre partiellement compense par `class_weight` sans degrader excessivement la calibration ;
- une baseline solide ici permettra de juger objectivement l'interet de ViT ou d'autres architectures plus couteuses.

## Sorties attendues

1. une vue claire de la distribution train/test ;
2. un controle du pretraitement et des resolutions source ;
3. une baseline multiclasse interpretable ;
4. une synthese des confusions critiques avant d'aller plus loin.

In [ ]:
## 1. Configuration & imports
import warnings; warnings.filterwarnings('ignore')

from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

import tensorflow as tf
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_class_weight

from src.utils.config import load_config
from src.preprocessing.image_loader import load_and_preprocess_image

# ── Config ──────────────────────────────────────────────────────────────────
cfg       = load_config('configs/brain_tumor_mri.yaml')
DATA_DIR  = Path(cfg['dataset_dir'])
TRAIN_DIR = DATA_DIR / cfg.get('training_subdir', 'Training')
TEST_DIR  = DATA_DIR / cfg.get('testing_subdir', 'Testing')
IMG_SIZE  = cfg.get('image_size', 224)
BATCH     = cfg.get('batch_size', 32)
CLASSES   = ['glioma', 'meningioma', 'no_tumor', 'pituitary']

print(f"Dataset dir  : {DATA_DIR}  |  existe = {DATA_DIR.exists()}")
print(f"Train dir    : {TRAIN_DIR.exists()}")
print(f"Test dir     : {TEST_DIR.exists()}")
print(f"Image size   : {IMG_SIZE}")
print(f"Batch size   : {BATCH}")
print(f"TensorFlow   : {tf.__version__}")


In [ ]:
## 2. Inspection de l'arborescence train/test
def count_by_class(base_dir: Path) -> dict:
    if not base_dir.exists():
        return {}
    return {
        cls.name: sum(1 for p in cls.rglob('*') if p.is_file())
        for cls in sorted(base_dir.iterdir()) if cls.is_dir()
    }

train_counts = count_by_class(TRAIN_DIR)
test_counts  = count_by_class(TEST_DIR)

if train_counts:
    df_counts = pd.DataFrame({'Train': train_counts, 'Test': test_counts}).fillna(0).astype(int)
    df_counts['Total'] = df_counts.sum(axis=1)
    df_counts.loc['TOTAL'] = df_counts.sum()
    print("Distribution par classe :")
    print(df_counts.to_string())
    print()
    # Déséquilibre
    print("Ratios relatifs (Train) :")
    total_train = sum(train_counts.values())
    for cls, n in sorted(train_counts.items()):
        print(f"  {cls:<15} {n:>5} images  ({100*n/total_train:.1f}%)")
else:
    print("⚠ Dataset non disponible — lance download_brain_mri_dataset.sh")


In [ ]:
## 3. Distribution des classes — visualisation
if train_counts:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

    # Barres
    sorted_classes = sorted(train_counts.keys())
    ax = axes[0]
    bars = ax.bar(sorted_classes,
                  [train_counts.get(c, 0) for c in sorted_classes],
                  color=colors)
    ax.set_title('Train — images par classe', fontweight='bold')
    ax.set_ylabel('Nombre d\'images')
    for bar, val in zip(bars, [train_counts.get(c, 0) for c in sorted_classes]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                str(val), ha='center', va='bottom', fontsize=10)

    # Camembert
    ax2 = axes[1]
    sizes = [train_counts.get(c, 0) for c in sorted_classes]
    ax2.pie(sizes, labels=sorted_classes, colors=colors, autopct='%1.1f%%',
            startangle=90)
    ax2.set_title('Répartition Train', fontweight='bold')

    plt.suptitle('Brain Tumor MRI — Distribution des classes', fontsize=13)
    plt.tight_layout()
    plt.show()


In [ ]:
## 4. Grille d'images brutes par classe
if TRAIN_DIR.exists():
    fig, axes = plt.subplots(4, 5, figsize=(15, 13))
    class_dirs = sorted(TRAIN_DIR.iterdir())
    colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

    for row_idx, cls_dir in enumerate(class_dirs[:4]):
        imgs = sorted(cls_dir.glob('*'))[:5]
        for col_idx, img_path in enumerate(imgs):
            ax = axes[row_idx][col_idx]
            img = Image.open(img_path).convert('L')
            ax.imshow(img, cmap='gray')
            ax.axis('off')
            if col_idx == 0:
                ax.set_ylabel(cls_dir.name, color=colors[row_idx],
                              fontsize=11, fontweight='bold', rotation=90, labelpad=10)
            ax.set_title(f'{img.size[0]}×{img.size[1]}', fontsize=7)

    plt.suptitle('Échantillons IRM — 4 classes (5 images par classe)', fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("Dataset absent.")


In [ ]:
## 5. Vérification du pipeline de prétraitement IRM
if TRAIN_DIR.exists():
    sample_class = sorted(TRAIN_DIR.iterdir())[0]
    sample_path  = sorted(sample_class.glob('*'))[0]

    img_pil = Image.open(sample_path).convert('RGB')
    arr     = load_and_preprocess_image(sample_path, image_size=IMG_SIZE)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(img_pil, cmap='gray')
    axes[0].set_title(f'Original\n{img_pil.size[0]}×{img_pil.size[1]}')
    axes[0].axis('off')

    axes[1].imshow(arr)
    axes[1].set_title(f'Prétraitée\nshape={arr.shape}')
    axes[1].axis('off')

    axes[2].hist(arr[:, :, 0].ravel(), bins=50, color='#8e44ad', alpha=0.7, label='R')
    axes[2].hist(arr[:, :, 1].ravel(), bins=50, color='#27ae60', alpha=0.7, label='G')
    axes[2].hist(arr[:, :, 2].ravel(), bins=50, color='#e74c3c', alpha=0.7, label='B')
    axes[2].set_title(f'Histogramme RGB\nmin={arr.min():.2f}, max={arr.max():.2f}')
    axes[2].legend()

    plt.suptitle(f'{sample_class.name} — {sample_path.name}', fontweight='bold')
    plt.tight_layout()
    plt.show()


In [ ]:
## 6. Datasets TensorFlow + class_weights
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(base_dir: Path, shuffle: bool = False):
    if not base_dir.exists():
        return None
    ds = tf.keras.utils.image_dataset_from_directory(
        base_dir,
        label_mode='int',
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH,
        shuffle=shuffle,
        seed=42,
    )
    return ds

# Normalisation EfficientNet
def preprocess(img, lbl):
    img = tf.cast(img, tf.float32)
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return img, lbl

train_ds_raw = make_ds(TRAIN_DIR, shuffle=True)
test_ds_raw  = make_ds(TEST_DIR)

if train_ds_raw is not None:
    found_classes = train_ds_raw.class_names
    print("Classes trouvées :", found_classes)

    train_ds = train_ds_raw.map(preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
    test_ds  = test_ds_raw.map(preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE) if test_ds_raw else None

    # Calcul class_weights
    all_labels = np.concatenate([y.numpy() for _, y in train_ds_raw])
    unique_cls  = np.unique(all_labels)
    class_weights_arr = compute_class_weight('balanced', classes=unique_cls, y=all_labels)
    class_weight_dict = {int(c): float(w) for c, w in zip(unique_cls, class_weights_arr)}
    print("Class weights :", class_weight_dict)
else:
    train_ds = test_ds = None
    class_weight_dict = {}
    print("⚠ Dataset absent.")


## Synthese decisionnelle

Ce notebook a de la valeur uniquement s'il sert a trancher une question experimentale precise : la classification 4 classes est-elle deja defendable avec une baseline sobre, ou faut-il revoir le probleme, les donnees ou le protocole d'evaluation avant d'investir davantage ?

## Points a verifier avant d'aller plus loin

- s'assurer que le split respecte bien l'independance entre train et test ;
- inspecter la matrice de confusion pour identifier les couples de classes structurellement difficiles ;
- verifier que les exemples mal classes ne sont pas domines par des artefacts d'acquisition ou de cadrage ;
- conserver la trace des hyperparametres utilises pour rendre toute comparaison ulterieure recevable.

## Criteres go / no-go avant integration

- rapporter la performance par classe, pas seulement une accuracy globale ;
- exiger une revue des faux `no_tumor` et des confusions entre sous-types tumoraux ;
- ne retenir ce probleme pour une phase applicative que si la performance reste stable apres reexecution ;
- utiliser ce notebook comme reference de baseline pour juger objectivement ConvNeXt, ViT ou d'autres variantes.

## Limite assumee

Le depot current fournit le cadre, l'EDA et la logique d'entrainement, mais une validation pre-production serieuse demandera ensuite un protocole plus strict, idealement patient-wise, plus une revue qualitative des erreurs avec expertise domaine.